# Load & Push

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import datasets
from google.colab import drive

cache_path = "/content/huggingface_cache"
os.makedirs(cache_path, exist_ok=True)
os.environ['HF_HOME'] = cache_path

In [2]:
from huggingface_hub import login, hf_hub_download
from google.colab import userdata

if userdata.get('HF_TOKEN'):
  login(token=userdata.get('HF_TOKEN'))
  hf_hub_download(repo_id="sookiemonster/asrs-narratives", filename="rebalance_all.csv", repo_type="dataset",local_dir=".")
  hf_hub_download(repo_id="sookiemonster/asrs-narratives", filename="utils.py", repo_type="dataset",local_dir=".")

train_ds = datasets.load_dataset("sookiemonster/asrs-narratives", split="train")
valid_ds = datasets.load_dataset("sookiemonster/asrs-narratives", split="validation")
test_ds = datasets.load_dataset("sookiemonster/asrs-narratives", split="test")

rebalance_all.csv:   0%|          | 0.00/23.7M [00:00<?, ?B/s]

utils.py: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/9.74M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/4.09M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/9.38M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10360 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4441 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9868 [00:00<?, ? examples/s]

In [3]:
rebalance_file = "rebalance_all.csv"
df = pd.read_csv(rebalance_file, index_col=0)
df = df.sample(random_state=42, frac=1.0)
df

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
acn,,,
1777008,weather,I was on an IFR flight plan to ZZZ descending ...,NaN
1751568,humanfactors,Operating Flight ABC ZZZ to ZZZ1. Neither the ...,NaN
1688615,procedure,During taxi; as I take my position as B [fligh...,NaN
1284042,environment-nonweatherrelated,We were ferrying an aircraft to Africa. On an ...,NaN
1666727,procedure,Working Local East; TRACON called the SUP and ...,NaN
...,...,...,...
1269926,companypolicy,[We] received a call from [operations] warning...,NaN
1537257,procedure,I was performing the duties of pilot monitorin...,NaN
1481252,procedure,While working traffic; I noticed Aircraft X tu...,During our climb on the departure out of we cr...


In [4]:
def parse_narrative_list(narratives:list[str]):
  return "\n".join([f"Narrative {idx+1} - '{narratives[idx]}'" for idx in range(len(narratives)) if pd.notna(narratives[idx])])

def structure_raw_df(df:pd.DataFrame):
  raw = df[['Report_1_Narrative', 'Report_2_Narrative']].copy().apply(
      lambda x: parse_narrative_list(x.to_list()),
      axis=1
  )
  narratives =  pd.DataFrame(raw, columns=['text'])
  narratives = narratives.join(df.Assessments_Primary_Problem)
  narratives = narratives.rename({'Assessments_Primary_Problem' : 'label'}, axis=1)
  return narratives

augment_df = structure_raw_df(df)
augment_df.head()

,text,label
acn,,
1777008,Narrative 1 - 'I was on an IFR flight plan to ...,weather
1751568,Narrative 1 - 'Operating Flight ABC ZZZ to ZZZ...,humanfactors
1688615,Narrative 1 - 'During taxi; as I take my posit...,procedure
1284042,Narrative 1 - 'We were ferrying an aircraft to...,environment-nonweatherrelated
1666727,Narrative 1 - 'Working Local East; TRACON call...,procedure


In [6]:
train_ds.features

{'acn': Value('int64'),
 'text': Value('string'),
 'label': ClassLabel(names=['aircraft', 'airport', 'airspacestructure', 'ambiguous', 'atcequipment/navfacility/buildings', 'chartorpublication', 'companypolicy', 'environment-nonweatherrelated', 'equipment/tooling', 'humanfactors', 'incorrect/notinstalled/unavailablepart', 'logbookentry', 'manuals', 'mel', 'procedure', 'softwareandautomation', 'staffing', 'weather'])}

In [27]:
existing_acns = set(train_ds['acn']) | set(valid_ds['acn']) | set(test_ds['acn'])
filtered = augment_df[~augment_df.index.isin(existing_acns)]
print(f"Removed {len(augment_df) - len(filtered)} duplicates.")

Removed 334 duplicates.


In [28]:
filtered.label.value_counts()

,count
label,
procedure,4621
humanfactors,2016
weather,1405
companypolicy,1226
environment-nonweatherrelated,873
chartorpublication,560
airspacestructure,529
atcequipment/navfacility/buildings,469
equipment/tooling,316


In [29]:
from datasets import Dataset, concatenate_datasets, DatasetDict

augment_dataset = Dataset.from_pandas(filtered)
augment_dataset = augment_dataset.cast_column('label', train_ds.features['label'])
augment_dataset.features

Casting the dataset:   0%|          | 0/12632 [00:00<?, ? examples/s]

{'text': Value('string'),
 'label': ClassLabel(names=['aircraft', 'airport', 'airspacestructure', 'ambiguous', 'atcequipment/navfacility/buildings', 'chartorpublication', 'companypolicy', 'environment-nonweatherrelated', 'equipment/tooling', 'humanfactors', 'incorrect/notinstalled/unavailablepart', 'logbookentry', 'manuals', 'mel', 'procedure', 'softwareandautomation', 'staffing', 'weather']),
 'acn': Value('int64')}

In [30]:
updated_train = concatenate_datasets([train_ds, augment_dataset])

updated_ds = DatasetDict({
    "train": updated_train,
    "validation": valid_ds,
    "test": test_ds
})

updated_ds

DatasetDict({
    train: Dataset({
        features: ['acn', 'text', 'label'],
        num_rows: 22992
    })
    validation: Dataset({
        features: ['acn', 'text', 'label'],
        num_rows: 4441
    })
    test: Dataset({
        features: ['acn', 'text', 'label'],
        num_rows: 9868
    })
})

In [32]:
updated_ds.push_to_hub("sookiemonster/asrs-narratives-rebalance")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/23 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  40%|###9      | 9.09MB / 22.8MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  98%|#########8| 9.20MB / 9.38MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/sookiemonster/asrs-narratives-rebalance/commit/dba47e674cbbd9b7b5a8242c6bb7ac4bfb57225b', commit_message='Upload dataset', commit_description='', oid='dba47e674cbbd9b7b5a8242c6bb7ac4bfb57225b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/sookiemonster/asrs-narratives-rebalance', endpoint='https://huggingface.co', repo_type='dataset', repo_id='sookiemonster/asrs-narratives-rebalance'), pr_revision=None, pr_num=None)